
# Interactive Workshop – Self-Supervised Hydrophone Anomaly Detection (Colab Edition)

This notebook guides you through a hands-on exploration of self‑supervised anomaly detection for hydrophone data. You’ll set up your environment, fetch and explore a mini dataset, train both self‑supervised and supervised models, compare their performance, and even test your own recordings.

**Workshop specifics:**

- **Duration:** ~3 hours, including breaks
- **Platform:** Google Colab (GPU runtime recommended)
- **Audience:** Ocean‑science professionals new to machine learning
- **Host:** Spencer Bialek
- **Date:** October 2025

Run each cell in order. Feel free to adjust parameters and explore further! All commands assume execution in a Colab environment.



## 1. Pre‑workshop setup

1. **Sign in** with your Google account.
2. Set the runtime to **GPU** via **Runtime → Change runtime type → GPU**.
3. Optionally mount your Google Drive to save data and models across sessions.

The cell below checks CUDA availability and can mount Google Drive if you uncomment it.


In [ ]:
#@title Check GPU availability and mount Google Drive (optional)
import torch

print(f"CUDA available: {torch.cuda.is_available()}")

# Uncomment these lines to mount your Google Drive for persistent storage
# from google.colab import drive
# drive.mount('/content/drive')



## 2. Clone the repository and set up environment

1. Clone the repo (tutorial branch).
2. Run the setup script to pin Torch and apply protobuf/s3prl fix.
3. Restart the runtime (this clears state; re-run from top afterwards).
4. Install mamba-ssm (GPU only).
5. Install remaining repo dependencies.


In [ ]:
%%bash
#@title Clone the repository (tutorial branch)
REPO_URL="https://github.com/Spiffical/selfsupervision_anomalies_onc.git"
REPO_DIR="selfsupervision_anomalies_onc"
BRANCH="${BRANCH:-tutorial}"

if [ ! -d "$REPO_DIR/.git" ]; then
    echo "Cloning $REPO_URL (branch: $BRANCH) ..."
    git clone -b "$BRANCH" --single-branch "$REPO_URL" "$REPO_DIR" \n      || { echo "Falling back to default branch..."; git clone "$REPO_URL" "$REPO_DIR"; }
else
    echo 'Repository already cloned.'
fi

# Ensure we are on the desired branch if it exists
cd "$REPO_DIR"
git fetch origin "$BRANCH" || true
git checkout "$BRANCH" 2>/dev/null || echo "Branch $BRANCH not found; staying on default."
git pull --ff-only || true
cd ..


In [ ]:
%%bash
#@title 🔧 Setup Colab environment via repo scripts (Torch pin + protobuf/s3prl fix)
python selfsupervision_anomalies_onc/colab/patch_requirements_for_colab.py selfsupervision_anomalies_onc || true
bash selfsupervision_anomalies_onc/colab/install_torch.sh


In [ ]:
#@title ♻️ Restart runtime to activate new Torch stack
import os, signal
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
%%bash
#@title 🚀 Install mamba-ssm (prebuilt wheel; GPU only)
python selfsupervision_anomalies_onc/colab/install_mamba.py


In [ ]:
#@title ✅ Verify local causal_conv1d shim and Mamba import
import sys, pathlib
repo_path = pathlib.Path('selfsupervision_anomalies_onc').resolve()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print('Repo on sys.path:', repo_path.exists(), str(repo_path))

import causal_conv1d as ccv
print('causal_conv1d module:', ccv.__file__)

import torch, mamba_ssm
print('mamba-ssm version:', mamba_ssm.__version__, '| CUDA available:', torch.cuda.is_available())

# Minimal functional check
if torch.cuda.is_available():
    try:
        from mamba_ssm.modules.mamba_simple import Mamba
        device = 'cuda'
        m = Mamba(d_model=64, d_state=16, d_conv=4, expand=2).to(device)
        x = torch.randn(1, 128, 64, device=device)
        y = m(x)
        print('Mamba forward OK:', tuple(y.shape))
    except Exception as e:
        print('Mamba CUDA test failed:', e)
else:
    print('Skipping Mamba test on CPU-only runtime. Enable GPU in Colab.')


In [ ]:
%%bash
#@title Install dependencies
cd selfsupervision_anomalies_onc

# Run the install script; patched requirements should prevent version conflicts
bash install_deps.sh

cd ..



## 3. Data download and preparation

Download a mini dataset of spectrograms for the workshop. Replace `<URL_TO_SAMPLE_ZIP>` with a real URL or mount your Google Drive and set the path to your own data. The files will be extracted into `spectrogram_data`.


In [ ]:
%%bash
#@title Download and extract sample dataset
SAMPLE_ZIP_URL="<URL_TO_SAMPLE_ZIP>"  # Replace with the actual URL
SAMPLE_ZIP_PATH="sample_spectrograms.zip"
DATA_DIR="spectrogram_data"

if [ ! -f "$SAMPLE_ZIP_PATH" ] && [ "$SAMPLE_ZIP_URL" != "<URL_TO_SAMPLE_ZIP>" ]; then
    echo 'Downloading sample data...'
    wget -O "$SAMPLE_ZIP_PATH" "$SAMPLE_ZIP_URL"
else
    echo 'Sample zip already present or URL not set.'
fi

if [ ! -d "$DATA_DIR" ] && [ -f "$SAMPLE_ZIP_PATH" ]; then
    echo 'Extracting data...'
    unzip "$SAMPLE_ZIP_PATH" -d "$DATA_DIR"
else
    echo 'Data directory exists or zip missing.'
fi



## 4. Visualisation and audio playback

Use this cell to inspect a few spectrograms and listen to the corresponding audio. Adjust the directory names if your data structure differs.


In [ ]:
#@title Visualize spectrograms and play audio
import os, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

spec_dir = os.path.join('spectrogram_data', 'spectrograms')
audio_dir = os.path.join('spectrogram_data', 'audio')

if os.path.isdir(spec_dir) and os.path.isdir(audio_dir):
    files = sorted([f for f in os.listdir(spec_dir) if f.endswith(('.png', '.npy'))])
    print(f'Found {len(files)} spectrograms.')
    for i in range(min(3, len(files))):
        fname = files[i]
        stem, ext = os.path.splitext(fname)
        spec_path = os.path.join(spec_dir, fname)
        audio_path = os.path.join(audio_dir, stem + '.wav')
        if ext == '.npy':
            spec = np.load(spec_path)
        else:
            spec = plt.imread(spec_path)
        plt.figure(figsize=(8, 3))
        plt.imshow(spec, aspect='auto', origin='lower', cmap='magma')
        plt.title(f'Spectrogram: {fname}')
        plt.xlabel('Time bins')
        plt.ylabel('Frequency bins')
        plt.colorbar(label='Amplitude (dB)')
        plt.show()
        if os.path.exists(audio_path):
            print(f'Playing audio for {stem}...')
            display(Audio(filename=audio_path))
        else:
            print(f'No audio found for {stem}.')
else:
    print('Spectrogram or audio directory missing.')



## 5. Train the self‑supervised model

Pre‑train the SSAMBA model with a masked patch joint objective and then fine‑tune it for classification. Adjust `batch-size`, `epochs`, and dataset path as needed. If your Colab runtime struggles with custom CUDA kernels, rely on the built‑in `causal_conv1d` shim provided by this repo (activated once the repo is on `sys.path`).


In [ ]:
%%bash
#@title Pre-train SSAMBA
DATASET_H5="/content/spectrogram_data/dataset.h5"  # Update if necessary
EXP_DIR="/content/ssamba_experiments"
mkdir -p "$EXP_DIR"

python selfsupervision_anomalies_onc/src/run_amba_spectrogram.py     --dataset "$DATASET_H5"     --task pretrain_joint     --batch-size 8     --epochs 3     --train-ratio 0.8     --exp-dir "$EXP_DIR"


In [ ]:
%%bash
#@title Fine-tune SSAMBA
PRETRAINED_PATH="<PATH_TO_PRETRAINED_MODEL>.pth"  # Replace with your pre-trained model path

python selfsupervision_anomalies_onc/src/run_amba_spectrogram.py     --dataset "$DATASET_H5"     --task ft_avgtok     --pretrained-path "$PRETRAINED_PATH"     --batch-size 8     --epochs 3     --train-ratio 0.8     --exp-dir "$EXP_DIR"



## 6. Supervised baseline: a small CNN

For comparison, train a lightweight CNN on your dataset. This baseline helps illustrate the benefits of self‑supervised learning. The placeholder dataset here is synthetic; replace it with your own spectrogram tensors.


In [ ]:
#@title Define and train a small CNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

# Replace this with actual HDF5 loading code
def load_dataset():
    num_samples = 100
    num_classes = 2
    X = np.random.randn(num_samples, 1, 128, 128).astype(np.float32)
    y = np.random.randint(0, num_classes, size=num_samples)
    return X, y

class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

def prepare_dataloaders(batch_size=16, val_ratio=0.1, test_ratio=0.1):
    X, y = load_dataset()
    dataset = TensorDataset(torch.tensor(X), torch.tensor(y))
    total = len(dataset)
    n_val = int(val_ratio * total)
    n_test = int(test_ratio * total)
    n_train = total - n_val - n_test
    train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])
    return DataLoader(train_set, batch_size=batch_size, shuffle=True), DataLoader(val_set, batch_size=batch_size), DataLoader(test_set, batch_size=batch_size)

def train_model(model, train_loader, val_loader, epochs=3, lr=1e-3):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(Xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * Xb.size(0)
        train_loss /= len(train_loader.dataset)
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds = model(Xb)
                loss = loss_fn(preds, yb)
                val_loss += loss.item() * Xb.size(0)
                correct += (preds.argmax(dim=1) == yb).sum().item()
                total += yb.size(0)
        val_loss /= len(val_loader.dataset)
        print(f'Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val Acc={correct/total:.4f}')
    return model

def evaluate_model(model, test_loader):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device)
            logits = model(Xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            y_score.extend(probs)
            y_true.extend(yb.numpy())
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    auc_roc = roc_auc_score(y_true, y_score)
    auc_pr = average_precision_score(y_true, y_score)
    cm = confusion_matrix(y_true, (y_score > 0.5).astype(int))
    return auc_roc, auc_pr, cm

train_loader, val_loader, test_loader = prepare_dataloaders()
cnn_model = SmallCNN()
cnn_model = train_model(cnn_model, train_loader, val_loader, epochs=3)
roc_auc, pr_auc, cm = evaluate_model(cnn_model, test_loader)
print(f'Supervised CNN → ROC AUC: {roc_auc:.3f}, AUPRC: {pr_auc:.3f}')
print('Confusion Matrix:
', cm)



## 7. Evaluation: ROC and Precision‑Recall curves

Visualize and compare the performance of the self‑supervised model versus the supervised baseline. Replace the random arrays with your actual scores and labels.


In [ ]:
#@title Plot ROC and PR curves
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

# Replace these with actual results
y_true = np.random.randint(0, 2, 100)
ssl_scores = np.random.rand(100)
cnn_scores = np.random.rand(100)

fpr_ssl, tpr_ssl, _ = roc_curve(y_true, ssl_scores)
fpr_cnn, tpr_cnn, _ = roc_curve(y_true, cnn_scores)
prec_ssl, rec_ssl, _ = precision_recall_curve(y_true, ssl_scores)
prec_cnn, rec_cnn, _ = precision_recall_curve(y_true, cnn_scores)

roc_auc_ssl = auc(fpr_ssl, tpr_ssl)
roc_auc_cnn = auc(fpr_cnn, tpr_cnn)
pr_auc_ssl = auc(rec_ssl, prec_ssl)
pr_auc_cnn = auc(rec_cnn, prec_cnn)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(fpr_ssl, tpr_ssl, label=f'SSL (AUC={roc_auc_ssl:.2f})')
plt.plot(fpr_cnn, tpr_cnn, label=f'CNN (AUC={roc_auc_cnn:.2f})', linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(rec_ssl, prec_ssl, label=f'SSL (AUPRC={pr_auc_ssl:.2f})')
plt.plot(rec_cnn, prec_cnn, label=f'CNN (AUPRC={pr_auc_cnn:.2f})', linestyle='--')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()

plt.tight_layout()
plt.show()



## 8. Try your own audio file

Upload a `.wav` file and compute the anomaly probability with your fine‑tuned SSAMBA model. Ensure that `PRETRAINED_PATH` in Section 6 points to a valid checkpoint.


In [ ]:
#@title Upload and evaluate your own .wav file
import torch
from google.colab import files
import librosa
import numpy as np

# Helper to preprocess audio to mel-spectrogram
def preprocess_audio(path):
    y, sr = librosa.load(path, sr=None)
    spec = librosa.feature.melspectrogram(y, sr=sr, n_fft=1024, hop_length=512, n_mels=128)
    spec_db = librosa.power_to_db(spec, ref=np.max)
    spec_norm = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min())
    return spec_norm[np.newaxis, ...].astype(np.float32)

uploaded = files.upload()
for fname in uploaded.keys():
    print('Processing', fname)
    spec = preprocess_audio(fname)
    tensor = torch.tensor(spec).unsqueeze(0)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tensor = tensor.to(device)
    # Load the fine-tuned model (update PRETRAINED_PATH accordingly)
    model_path = PRETRAINED_PATH
    from selfsupervision_anomalies_onc.src.ssamba.models.model_utils import load_audio_model
    model = load_audio_model(model_path).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        prob = torch.softmax(logits, dim=1)[0, 1].item()
    print(f'Anomaly probability: {prob:.3f}')



## 9. Save your models

Copy your experiment outputs to Google Drive for safekeeping. Make sure you have mounted your drive before running this cell.


In [ ]:
%%bash
#@title Save experiments to Google Drive
SOURCE_DIR="/content/ssamba_experiments"
TARGET_DIR="/content/drive/MyDrive/ssamba_workshop"

if [ -d "$SOURCE_DIR" ]; then
    mkdir -p "$TARGET_DIR"
    cp -r "$SOURCE_DIR" "$TARGET_DIR/"
    echo "Copied experiments to $TARGET_DIR."
else
    echo "No experiments directory found."
fi



## 10. Next steps and resources

- Try other SSAMBA tasks (`pretrain_mpc`, `pretrain_mpg`) and fine‑tuning strategies (`ft_cls`).
- Explore hyperparameter tuning: batch size, learning rate, and number of epochs can all improve performance.
- Use the interactive labeling tool located in `tools/labeling/run.py` to create more labeled data.
- For larger-scale experiments, consider moving to DRAC or another machine with a full CUDA toolkit installed.

### Further reading

- Self‑supervised pre‑training and fine‑tuning tasks【884751655954375†L247-L334】
- Data download and preparation for ONC spectrograms【884751655954375†L196-L220】
- The SSAMBA repository README for details on tasks and model usage【884751655954375†L247-L334】

Thank you for participating!
